# Quality Estimation for Machine Translation

**NLPAICS 2026 Summer School — The Paradigm Shift** · Day 2 · Tuesday 16 June 2026

**Lecturer:** Tharindu Ranasinghe

> Before running anything, make sure the kernel is **NLPAICS 04** (menu: *Kernel → Change Kernel*). It should already be selected.

## 0 · Environment check

In [ ]:
# --- Environment check: run this cell first ---------------------------------
# It verifies you are on this lesson's kernel and that the GPU is visible.
import sys

assert ".venv" in sys.executable, (
    "Wrong kernel! In the menu choose: Kernel > Change Kernel > 'NLPAICS 04"
)
print("Kernel OK:", sys.executable)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (fine if this lesson doesn't need it)")


# Machine Translation & Quality Estimation — Practical Session

In this session we follow the natural life-cycle of a machine translation:

1. **Translate** — we play with two open, multilingual MT engines on a set of *deliberately difficult* Spanish sentences and watch them struggle. This is where the **importance of good translation** becomes visible.
2. **Evaluate (with a reference)** — we score outputs with the classic reference-based metrics **BLEU** and **BERTScore**, and see what they can and cannot capture.
3. **Estimate quality (without a reference)** — in the real world (live chat, news feeds, on-the-fly translation) we almost never have a human reference. That is why we turn to **Quality Estimation (QE)** with **COMET-Kiwi** and **GEMBA-MQM**.

Everything stays in the **Spanish &harr; English** pair so the three parts tell one story.

# Part 1 — Machine Translation Models

Machine Translation (MT) is the task of automatically translating text from one language to another. Neural MT (NMT) is today's dominant paradigm, and although early systems were built for a single language pair, modern NMT models are naturally **multilingual**.

Let's play with two widely used open multilingual systems, **M2M100** and **NLLB**, and translate Spanish into English.

### Our (difficult) Spanish source sentences

We start with one **easy** sentence as a control, then a set of **hard** ones. Each hard sentence is chosen because it breaks a naive word-by-word translation — this is precisely why translation quality matters and cannot be taken for granted.

| # | Spanish source | Why it is hard | A good English translation |
|---|---|---|---|
| 0 | *La biblioteca abre a las nueve de la mañana.* | easy control | The library opens at nine in the morning. |
| 1 | *Tirar la casa por la ventana.* | idiom — literally *throw the house out of the window* | To spare no expense. |
| 2 | *Estoy en el quinto pino.* | idiom — literally *I'm in the fifth pine* | I'm in the middle of nowhere. |
| 3 | *No tengo pelos en la lengua.* | idiom — literally *I have no hairs on my tongue* | I don't mince my words. |
| 4 | *Cuesta un ojo de la cara.* | idiom — literally *it costs an eye of the face* | It costs an arm and a leg. |
| 5 | *Dejó el libro en el banco.* | lexical ambiguity — *banco* = bench **or** bank | She left the book on the bench. |

In [ ]:
source_sentences = [
    "La biblioteca abre a las nueve de la mañana.",   # 0 easy control
    "Tirar la casa por la ventana.",                  # 1 idiom
    "Estoy en el quinto pino.",                       # 2 idiom
    "No tengo pelos en la lengua.",                   # 3 idiom
    "Cuesta un ojo de la cara.",                      # 4 idiom
    "Dejó el libro en el banco.",                     # 5 ambiguity (bench / bank)
]

## M2M100

M2M100 (*Beyond English-Centric Multilingual Machine Translation*, Fan et al.) is a multilingual encoder-decoder model. The target language is forced as the first generated token via `forced_bos_token_id`.

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

m2m_model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
m2m_tok   = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
m2m_tok.src_lang = "es"

def m2m_translate(text, tgt="en"):
    enc = m2m_tok(text, return_tensors="pt")
    gen = m2m_model.generate(**enc, forced_bos_token_id=m2m_tok.get_lang_id(tgt))
    return m2m_tok.batch_decode(gen, skip_special_tokens=True)[0]

for s in source_sentences:
    print("ES:", s)
    print("EN:", m2m_translate(s), "\n")

## NLLB

NLLB (*No Language Left Behind*, Costa-jussà et al.) covers 200 languages. Unlike M2M100 it uses BCP-47 codes (Spanish = `spa_Latn`, English = `eng_Latn`). The small helper below reads the target-language id in a way that works across `transformers` versions.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

nllb_tok   = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="spa_Latn")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

def nllb_lang_id(tok, code):
    # Newer transformers dropped tokenizer.lang_code_to_id; fall back gracefully.
    if hasattr(tok, "lang_code_to_id"):
        return tok.lang_code_to_id[code]
    return tok.convert_tokens_to_ids(code)

def nllb_translate(text, tgt="eng_Latn"):
    enc = nllb_tok(text, return_tensors="pt")
    gen = nllb_model.generate(**enc,
                              forced_bos_token_id=nllb_lang_id(nllb_tok, tgt),
                              max_length=60)
    return nllb_tok.batch_decode(gen, skip_special_tokens=True)[0]

for s in source_sentences:
    print("ES:", s)
    print("EN:", nllb_translate(s), "\n")

### Compare the two engines side by side

In [ ]:
print(f"{'Spanish source':40s} | {'M2M100':40s} | {'NLLB':40s}")
print("-" * 126)
for s in source_sentences:
    print(f"{s:40s} | {m2m_translate(s):40s} | {nllb_translate(s):40s}")

**Discuss.** Notice how the idioms (1–4) tend to come out *literal* or only partly idiomatic, and how the ambiguous *banco* in sentence 5 forces the engine to **guess** between *bench* and *bank* with no way to ask for clarification. The two engines also disagree with each other.

This raises the obvious question: **different engines give different outputs — so which one is best?** To answer that, we need a way to *measure* translation quality.

### Your turn

Write a Spanish sentence you think will be hard (an idiom, a pun, an ambiguous word, a long subordinate clause) and translate it with both engines. Did either of them get it right?

In [ ]:
my_sentence = "..."   # <- your difficult Spanish sentence here

print("M2M100:", m2m_translate(my_sentence))
print("NLLB  :", nllb_translate(my_sentence))

# Part 2 — Machine Translation Evaluation (with a reference)

There are two ways to judge an MT system. **Human evaluation** is the gold standard but is slow and expensive. **Automatic evaluation** uses metrics that compare the MT output against one or more **human reference translations**. Let's look at the two most common reference-based metrics.

## BLEU

BLEU compares the n-grams of the candidate translation against a human reference and returns a score: the more n-grams they share, the higher the score. It is fast, language-independent and extremely widely used — but it only counts **surface word overlap**, so it is blind to meaning, to synonyms, and (for unigrams) to word order.

In [ ]:
import nltk
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu

def bleu(ref, gen):
    """Pairwise BLEU using the NLTK implementation (as in the original tutorial)."""
    ref_bleu = [[r.split()] for r in ref]
    gen_bleu = [g.split() for g in gen]
    cc = SmoothingFunction()
    return corpus_bleu(ref_bleu, gen_bleu, weights=(0, 1, 0, 0), smoothing_function=cc.method4)

**A faithful but non-literal translation.** Both sentences mean the same thing, yet BLEU drops because *greatest/biggest* and *time/era* are different surface words.

In [ ]:
Reference = "Climate change is one of the greatest challenges of our time."
Candidate = "Climate change is one of the biggest challenges of our era."
print("BLEU:", bleu([Reference], [Candidate]))

**An idiom translated literally.** This is what an engine produced for *Cuesta un ojo de la cara.* It is wrong, and BLEU correctly gives it a very low score — **but only because we happened to have the human reference**.

In [ ]:
Reference = "It costs an arm and a leg."
Candidate = "It costs an eye of the face."
print("BLEU:", bleu([Reference], [Candidate]))

**Word order.** BLEU (at the unigram level) gives these two the same score even though the second one means something completely different — a classic illustration of its blindness to order.

In [ ]:
Reference = "The guard arrived late because of the rain."
Candidate = "The rain arrived late because of the guard."
print("BLEU:", bleu([Reference], [Candidate]))

## BERTScore

To get past pure surface overlap we can compare **meaning** instead of exact words. **BERTScore** embeds every token of the candidate and the reference with a pretrained model (RoBERTa by default), matches each candidate token to its most similar reference token by cosine similarity (and vice versa), and aggregates those similarities into **precision, recall and F1**. Because it works on contextual embeddings, it rewards synonyms and paraphrases that BLEU would miss. It is **pure PyTorch** (built on HuggingFace `transformers`), so it sits on the same stack as COMET — no TensorFlow involved.

BERTScore rewards the *meaning-preserving* paraphrase that BLEU penalised, and still scores the nonsense candidate low. We report **F1** (the usual headline number).

In [ ]:
from bert_score import score

reference = ["Climate change is one of the greatest challenges of our time."]

good = ["Climate change is one of the biggest challenges of our era."]   # paraphrase
bad  = ["Climate change is a recipe for chocolate cake."]                # nonsense

_, _, f1_good = score(good, reference, lang="en")
_, _, f1_bad  = score(bad,  reference, lang="en")

print("BERTScore F1 (paraphrase):", round(f1_good.item(), 4))
print("BERTScore F1 (nonsense)  :", round(f1_bad.item(), 4))

## The catch: every metric so far needs a reference

BLEU and BERTScore — and even modern reference-based COMET — all require a **human reference translation** to compare against. That is fine in a research benchmark, but think about where MT is actually deployed:

* a live customer-support chat being translated as it is typed,
* a news feed translated into 50 languages the moment it is published,
* a translator's CAT tool flagging segments that need attention.

In none of these cases does a human reference exist — if it did, we would not need the MT in the first place. **This is the gap that Quality Estimation fills.**

# Part 3 — Quality Estimation (no reference)

The goal of **quality estimation (QE)** is to evaluate the quality of a translation **without having access to a reference**. Because QE only looks at the *source* and the *machine translation*, it can run in real time and be used to:

* select the best output when several engines are available;
* warn the end user about the reliability of the translation;
* decide whether a segment can be published as-is, needs post-editing, or must be re-translated.

QE can be done at **document**, **sentence** and **word** level. We will use two state-of-the-art models from the [COMET](https://github.com/Unbabel/COMET) framework:

| Granularity | Model | What it gives you |
|---|---|---|
| Sentence level | `Unbabel/wmt22-cometkiwi-da` (**COMET-Kiwi**) | one quality score in `[0, 1]` per segment |
| Word / span level | **GEMBA-MQM** prompt on a small LLM (Qwen2.5-3B-Instruct) | error spans tagged *minor / major / critical*, plus a severity-weighted score |

To make the errors easy to see, most of the Spanish &rarr; English translations below are **deliberately wrong**. Watch how the scores drop and how GEMBA-MQM points to exactly where the errors are.

## Setup

Part 3 uses the `unbabel-comet` package for sentence-level QE, and a small open LLM (via `transformers`) for the word-level part.

> **Runtime note.** COMET-Kiwi (&asymp; 2.3 GB) and the small LLM Qwen2.5-3B-Instruct (&asymp; 6 GB) both fit a free Colab T4. Use a **GPU runtime** (Colab: *Runtime &rarr; Change runtime type &rarr; GPU*).

`Unbabel/wmt22-cometkiwi-da` is a **gated model** (CC-BY-NC-SA, for non-commercial evaluation), so before downloading it you must:

1. While logged in to Hugging Face, **accept the licence** at https://huggingface.co/Unbabel/wmt22-cometkiwi-da ;
2. **Log in** below with an access token (https://huggingface.co/settings/tokens).

(The word-level model `Qwen/Qwen2.5-3B-Instruct` is **not** gated — no licence step needed.)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()          # paste a token with at least 'read' permission

# Non-interactive alternative:
# from huggingface_hub import login
# login(token="hf_xxx")

### Our Spanish &rarr; English QE examples

The **source** is Spanish, the **machine translation** is English. One translation is correct (control); the rest contain typical MT error types. For QE we pass the model **only** `src` and `mt` — never a reference.

| # | Error type | What is wrong |
|---|---|---|
| A | *none (control)* | faithful translation &rarr; should score **high** |
| B | negation flip | the MT inserts *not*, inverting the meaning |
| C | fluent but unfaithful | *agua* (water) is rendered as *wine* — fluent, completely wrong |
| D | multiple lexical errors | *gato* &rarr; dog, *calle* &rarr; river, *tranquilamente* &rarr; quickly |
| E | omission / under-translation | the time and the reason are dropped entirely |

In [ ]:
# Source is Spanish (src), machine translation is English (mt).
# COMET QE models take ONLY src + mt — no reference is provided.
examples = [
    {   # A — correct control
        "src": "El cambio climático es uno de los mayores desafíos de nuestro tiempo.",
        "mt":  "Climate change is one of the greatest challenges of our time.",
    },
    {   # B — negation flip (meaning inverted)
        "src": "Reducir estos conflictos es importante para la conservación.",
        "mt":  "Reducing these conflicts is not important for conservation.",
    },
    {   # C — fluent but unfaithful: agua (water) -> wine
        "src": "Los médicos recomiendan beber ocho vasos de agua al día.",
        "mt":  "Doctors recommend drinking eight glasses of wine a day.",
    },
    {   # D — several lexical errors: cat->dog, street->river, calmly->quickly
        "src": "El gato negro cruzó la calle tranquilamente.",
        "mt":  "The black dog crossed the river quickly.",
    },
    {   # E — omission: time + reason dropped
        "src": "La reunión se pospuso hasta el próximo martes debido al mal tiempo.",
        "mt":  "The meeting was postponed.",
    },
]

labels = ["A correct", "B negation", "C water->wine", "D lexical", "E omission"]
for lab, ex in zip(labels, examples):
    print(f"[{lab}]")
    print("  ES:", ex["src"])
    print("  EN:", ex["mt"])

## Sentence-level QE with COMET-Kiwi

`Unbabel/wmt22-cometkiwi-da` is a **reference-free** regression model (built on InfoXLM). It takes the source and the translation and returns one score in `[0, 1]`, where values close to **1** mean high quality and close to **0** mean poor quality. The COMET API is the same for every model: `download_model` &rarr; `load_from_checkpoint` &rarr; `model.predict(...)`.

In [ ]:
import os
os.environ["USE_TF"] = "0"          # ensure TensorFlow is never imported by transformers
os.environ["USE_FLAX"] = "0"

from comet import download_model, load_from_checkpoint
import torch

kiwi_path = download_model("Unbabel/wmt22-cometkiwi-da")
kiwi_model = load_from_checkpoint(kiwi_path)

In [ ]:
gpus = 1 if torch.cuda.is_available() else 0
kiwi_output = kiwi_model.predict(examples, batch_size=8, gpus=gpus)

print("COMET-Kiwi quality scores (reference-free)\n")
for lab, score in zip(labels, kiwi_output.scores):
    print(f"  {lab:15s}  {score:.4f}")

print(f"\nSystem (average) score: {kiwi_output.system_score:.4f}")

**What to look for.** The control (A) should sit near the top. The negation flip (B) and the *water &rarr; wine* substitution (C) are *fluent* English, yet COMET-Kiwi penalises them because they are unfaithful to the source and it does so **with no reference**, exactly the situation we are in when MT is deployed live.

### A real QE use case: ranking competing MT outputs

Recall the Part 1 question *which engine is best?* With QE we can answer it per sentence, with no reference, by scoring every candidate and keeping the highest.

In [ ]:
ranking_src = "La biblioteca abre a las nueve de la mañana de lunes a viernes."

candidates = [
    "The library opens at nine in the morning from Monday to Friday.",   # faithful
    "The library opens at nine in the morning from Monday to Sunday.",   # Friday -> Sunday
    "The bookshop closes at nine at night on weekends.",                 # many errors
]

ranking_data = [{"src": ranking_src, "mt": c} for c in candidates]
ranking_out = kiwi_model.predict(ranking_data, batch_size=8, gpus=gpus)

ranked = sorted(zip(candidates, ranking_out.scores), key=lambda x: x[1], reverse=True)

print("Source (ES):", ranking_src, "\n")
print("Candidates ranked by COMET-Kiwi:\n")
for rank, (cand, score) in enumerate(ranked, start=1):
    print(f"  {rank}. [{score:.4f}]  {cand}")

print(f"\nBest candidate selected by QE:\n  {ranked[0][0]}")

## Word-level QE with GEMBA-MQM (prompting an LLM to do QE)

A sentence score tells you *that* a translation is bad, not *where* or *why*. Instead of a dedicated word-level model, we can **prompt a small instruction-tuned LLM** to do the job. **GEMBA-MQM** (Kocmi & Federmann, 2023) is a *reference-free* method that asks the model to mark **error spans** in the translation and label each with an MQM **category** and **severity**:

* **minor** — technically an error, but it does not disrupt the flow or hinder comprehension;
* **major** — disrupts the flow, but the meaning is still understandable;
* **critical** — inhibits comprehension of the text.

The detected errors are turned into one segment score with the standard MQM weighting:

$$\text{score} = \max(-25,\; -25\,N_\text{critical} - 5\,N_\text{major} - N_\text{minor})$$

so **0 is perfect** and **-25 is the floor** (which stops the score running away when many errors are found). The original paper used GPT-4; here we use a small open model (an **SLM**) that fits a free Colab GPU, and we ask it to answer in **JSON** so the spans are easy to parse.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SLM = "Qwen/Qwen2.5-3B-Instruct"          # ungated, ~6 GB, fits a free Colab T4
tok = AutoTokenizer.from_pretrained(SLM)
slm = AutoModelForCausalLM.from_pretrained(
    SLM,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

The GEMBA-MQM prompt has three parts: a **system** instruction with the MQM error typology and severity definitions, three fixed **few-shot** examples (the prompt is language-agnostic, so the examples can be in any language pair), and finally our **source + machine translation**. We ask for JSON output and parse it into a list of errors.

In [ ]:
import json, re

SYSTEM = """You are an annotator for the quality of machine translation. Identify errors and assess the translation using the MQM framework.

Given the source text and the machine translation, find error spans in the translation and classify each by category and severity.

Categories: accuracy (addition, mistranslation, omission, untranslated text), fluency (character encoding, grammar, inconsistency, punctuation, register, spelling), style (awkward), terminology (inappropriate for context, inconsistent use), non-translation, other.

Severity:
- critical: errors that inhibit comprehension of the text.
- major: errors that disrupt the flow, but the meaning is still understandable.
- minor: technically errors, but they do not disrupt the flow or hinder comprehension.

Output ONLY a JSON array. Each element is an object with keys "error_span" (the exact substring of the translation), "category" and "severity". If there are no errors, output []."""

# Three fixed few-shot examples (language-agnostic, as in GEMBA-MQM)
FEWSHOT = [
    ("English source:\n```I like to eat apples.```\nGerman translation:\n```Ich esse gerne Birnen.```",
     '[{"error_span": "Birnen", "category": "accuracy/mistranslation", "severity": "major"}]'),
    ("German source:\n```Das Hotel war nicht sauber.```\nEnglish translation:\n```The hotel was clean.```",
     '[{"error_span": "clean", "category": "accuracy/mistranslation", "severity": "critical"}]'),
    ("English source:\n```Please send the report by Friday.```\nSpanish translation:\n```Por favor envia el informe para el viernes.```",
     '[{"error_span": "envia", "category": "fluency/spelling", "severity": "minor"}]'),
]

def build_messages(src, mt, src_lang="Spanish", tgt_lang="English"):
    msgs = [{"role": "system", "content": SYSTEM}]
    for u, a in FEWSHOT:
        msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user",
                 "content": f"{src_lang} source:\n```{src}```\n{tgt_lang} translation:\n```{mt}```"})
    return msgs

def run_slm(messages, max_new_tokens=512):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(slm.device)
    out = slm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def parse_errors(text):
    m = re.search(r"\[.*\]", text, re.DOTALL)        # grab the JSON array
    try:
        return json.loads(m.group(0)) if m else []
    except json.JSONDecodeError:
        return []

def mqm_score(errors):
    nc   = sum(e.get("severity") == "critical" for e in errors)
    nmaj = sum(e.get("severity") == "major" for e in errors)
    nmin = sum(e.get("severity") == "minor" for e in errors)
    return max(-25, -25 * nc - 5 * nmaj - 1 * nmin)   # 0 = perfect, -25 = worst

def gemba_mqm(src, mt, src_lang="Spanish", tgt_lang="English"):
    errors = parse_errors(run_slm(build_messages(src, mt, src_lang, tgt_lang)))
    return errors, mqm_score(errors)

Now run it on the same Spanish &rarr; English examples and render the flagged spans inline.

In [ ]:
def annotate(mt, spans):
    """Wrap each flagged span in <error severity=...> ... </error> inside the MT string."""
    if not spans:
        return mt + "    (no errors detected)"
    out = mt
    for s in spans:
        sev, txt = s.get("severity", "?"), s.get("text", "")
        if txt and txt in out:
            out = out.replace(txt, f"<error severity='{sev}'>{txt}</error>", 1)
    return out


span_examples = [examples[1], examples[2], examples[3]]   # B negation, C water->wine, D lexical
span_labels   = ["B negation", "C water->wine", "D lexical"]

for lab, ex in zip(span_labels, span_examples):
    errors, score = gemba_mqm(ex["src"], ex["mt"])
    spans = [{"text": e.get("error_span", ""), "severity": e.get("severity", "?")} for e in errors]
    print(f"=== {lab}  (GEMBA-MQM score = {score}) ===")
    print("  ES :", ex["src"])
    print("  MT :", annotate(ex["mt"], spans))
    for e in errors:
        print(f"       - [{e.get('severity')}] {e.get('category')}: {e.get('error_span')}")
    print()

The output is now **explainable** and reference-free: the negation flip should come back as *critical*, *wine* as a *critical* mistranslation of *agua*, and *dog / river / quickly* as the problem words in D. Note the change in spirit — COMET-Kiwi is a *purpose-built* QE model, whereas here we **prompt a general LLM** to behave like one, and the score is a penalty (0 best, -25 worst) rather than COMET-Kiwi's 0&ndash;1.

> An SLM's output is not perfectly deterministic and depends on model size. If the JSON comes back malformed or errors are missed, switch `SLM` to `Qwen/Qwen2.5-7B-Instruct` (more reliable, needs more VRAM).

## Exercise

Write **your own** Spanish source sentences and (wrong) English machine translations — try a negation, a wrong number, a dropped clause, or a fluent-but-unfaithful substitution. Then:

1. score them with **COMET-Kiwi** and check whether the scores match how bad you think each is;
2. run **GEMBA-MQM** on the worst one and see whether the error spans land where you expected.

In [ ]:
my_examples = [
    {"src": "...",   # your Spanish source
     "mt":  "..."},  # your (wrong) English translation
    # add more pairs here
]

# Sentence-level QE
my_kiwi = kiwi_model.predict(my_examples, batch_size=8, gpus=gpus)
for ex, score in zip(my_examples, my_kiwi.scores):
    print(f"[{score:.4f}]  {ex['mt']}")

# Word-level QE with GEMBA-MQM (uncomment once you have at least one real pair)
# for ex in my_examples:
#     errors, score = gemba_mqm(ex["src"], ex["mt"])
#     spans = [{"text": e.get("error_span", ""), "severity": e.get("severity", "?")} for e in errors]
#     print(score, annotate(ex["mt"], spans))

# Wrap-up

| | Needs a reference? | Output |
|---|---|---|
| BLEU | **yes** | n-gram overlap with a human translation |
| BERTScore | **yes** | embedding-similarity (P/R/F1) to a reference |
| COMET-Kiwi (`wmt22-cometkiwi-da`) | **no** | one quality score per segment |
| GEMBA-MQM (LLM prompt) | **no** | error spans with severities **+** a severity-weighted score |

The story of the session:

* **Translation is hard** — idioms and ambiguity in Part 1 show why quality cannot be assumed, and why different engines disagree.
* **Reference-based evaluation works but needs a human translation** — BLEU is blind to meaning, BERTScore fixes that, but both are unusable when no reference exists.
* **QE estimates quality from the source and the translation alone** — so it works live; COMET-Kiwi gives a score, and GEMBA-MQM goes further by prompting an LLM to tell you *which words* are wrong and *how badly*.